# Архитектура мультимодальной сети
При разработке мультимодальной архитектуры нужно учитывать два основных момента: 
* объединение энкодеров/декодеров разных модальностей и 
* объединение эмбеддингов разных модальностей.

Чтобы объединить две модели в одну, достаточно:
* Указать их в рамках одного класса nn.Module и проинициализировать.
* В методе forward() рассчитать эмбеддинги для каждой модели.

In [1]:
from transformers import AutoModel, AutoTokenizer 

import torch
import torch.nn as nn  
import timm

c:\Users\aseva\Desktop\MyEDU\YaDLE\YaDLE_gensim_vscode_stream_4\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class BaseMultimodalModel(nn.Module):
    def __init__(self,
                 text_model_name='bert-base-uncased',
                 image_model_name='resnet50'):
        super().__init__()

        # Текстовая ветка
        self.text_model = AutoModel.from_pretrained(text_model_name)

        # Визуальная ветка
        self.image_model = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0  # Возвращаем вектор признаков, а не классы
        )

    def forward(self, text_input, image_input):
        # эмбеддинги текста
        text_features = self.text_model(**text_input).last_hidden_state[:,  0, :]

        # эмбеддинги изображений
        image_features = self.image_model(image_input)
        return text_features, image_features

text_model = 'bert-base-uncased'
image_model = 'resnet50'
m = BaseMultimodalModel(text_model_name=text_model,
                        image_model_name=image_model)

# 2 примера текста и картинки для инференса
tk = AutoTokenizer.from_pretrained(text_model)
tokenized = tk(["text", "text2"],
               return_tensors="pt",
               padding='max_length',
               truncation=True)


# вместо самой картинки используем случайный тензор правильной размерности
# размерность тензора берём из конфига и распаковываем через `*`
img = torch.randn(2, *m.image_model.pretrained_cfg["input_size"])

t, i = m(tokenized, img)
t.shape, i.shape 